# 03 — Predict

**SDI Tehran Pipeline · Notebook 3 of 5**

## Purpose
Apply trained models to the unlabelled periphery segments. All scalers, PCA objects, and encoders are loaded from serialised files — nothing is refit here. This ensures predictions are made in the same feature space as the training data.

## Inputs
- `data/prediction_set.csv` — periphery segments with raw CH+INT features (no surveyed labels)
- `data/outputs/segments_pca_clustered.csv` — `whole_CH_2` cluster labels for all segments
- `models/pca_scaler.joblib` — StandardScaler fitted on training set CH+INT (NB01)
- `models/pca_model.joblib` — PCA(3) fitted on training set CH+INT (NB01)
- `models/model_scaler.joblib` — StandardScaler fitted on PC1-3 training set (NB02)
- `trained_models/clf_{attr}.joblib` — per-attribute MLPClassifier (NB02)
- `trained_models/le_{attr}.joblib` — per-attribute LabelEncoder (NB02)
- `trained_models/regressor.joblib` — multi-output MLPRegressor (NB02)

## Outputs
- `data/outputs/predictions_joined.csv` — 6 predicted attributes + `whole_CH_2` per periphery segment

## 0 · Config & imports

In [7]:
import sys, os
import numpy as np
import pandas as pd
import joblib

sys.path.insert(0, '..')
from config import (
    PREDICTION_SET_CSV, SEGMENTS_PCA_CLUSTERED_CSV,
    PREDICTIONS_JOINED_CSV,
    PCA_SCALER_PATH, PCA_MODEL_PATH, MODEL_SCALER_PATH,
    TRAINED_MODELS_DIR,
    PCA_INPUT_COLS,
    CATEGORICAL_TARGETS, CONTINUOUS_TARGETS,
    SEG_ID_COL
)

print("Config loaded.")
print(f"  Categorical targets : {CATEGORICAL_TARGETS}")
print(f"  Continuous targets  : {CONTINUOUS_TARGETS}")
print(f"  PCA input cols      : {len(PCA_INPUT_COLS)} (CH + INT across 8 radii)")

Config loaded.
  Categorical targets : ['policy_era', 'dilap_index', 'landuse']
  Continuous targets  : ['mean_area', 'perimeter', 'mean_pop']
  PCA input cols      : 16 (CH + INT across 8 radii)


## 1 · Load periphery segments

Load the raw prediction set and verify all CH+INT columns are present. `whole_CH_2` cluster labels are joined separately from `segments_pca_clustered.csv` — this is a visualisation attribute only and does not feed into prediction.

In [8]:
periphery = pd.read_csv(PREDICTION_SET_CSV)
print(f"Prediction set loaded: {len(periphery):,} segments")

missing = [c for c in PCA_INPUT_COLS if c not in periphery.columns]
assert not missing, f"Missing PCA input columns in prediction set: {missing}"

n_before = len(periphery)
periphery = periphery.dropna(subset=PCA_INPUT_COLS)
print(f"Dropped {n_before - len(periphery):,} rows with nulls in PCA input columns")

# Join whole_CH_2 for output only
pca_data = pd.read_csv(SEGMENTS_PCA_CLUSTERED_CSV, usecols=[SEG_ID_COL, 'whole_CH_2'])
periphery = periphery.merge(pca_data, on=SEG_ID_COL, how='left')
n_missing_cluster = periphery['whole_CH_2'].isna().sum()
if n_missing_cluster > 0:
    print(f"WARNING: {n_missing_cluster} segments had no match in segments_pca_clustered — check segment IDs.")
print(f"Periphery segments ready: {len(periphery):,}")

Prediction set loaded: 279,793 segments
Dropped 0 rows with nulls in PCA input columns
Periphery segments ready: 279,793


## 2 · Transform prediction set into training PCA space

**Critical:** The PCA scaler and PCA model were fitted on the **training set only** in NB01. Loading (not refitting) them here guarantees the periphery segments are projected into the same component space as the training data. Refitting on the prediction set would constitute data leakage and misalign the component axes.

In [9]:
pca_scaler   = joblib.load(PCA_SCALER_PATH)
pca_model    = joblib.load(PCA_MODEL_PATH)
model_scaler = joblib.load(MODEL_SCALER_PATH)

print(f"Loaded: {PCA_SCALER_PATH}")
print(f"Loaded: {PCA_MODEL_PATH}")
print(f"Loaded: {MODEL_SCALER_PATH}")

# Step 1: scale raw CH+INT features using training-set scaler
X_raw_scaled = pca_scaler.transform(periphery[PCA_INPUT_COLS].values)

# Step 2: project into PC1/PC2/PC3 using training-set PCA
X_pca = pca_model.transform(X_raw_scaled)

# Step 3: scale PCs using model scaler fitted on training set
X = model_scaler.transform(X_pca)

print(f"Feature matrix shape: {X.shape}")

Loaded: D:\SID\models\pca_scaler.joblib
Loaded: D:\SID\models\pca_model.joblib
Loaded: D:\SID\models\model_scaler.joblib
Feature matrix shape: (279793, 3)


## 3 · Predict categorical attributes

Each classifier and its label encoder are loaded individually. Predictions are inverse-transformed back to original class labels.

In [10]:
for attr in CATEGORICAL_TARGETS:
    clf_path = os.path.join(TRAINED_MODELS_DIR, f'clf_{attr}.joblib')
    le_path  = os.path.join(TRAINED_MODELS_DIR, f'le_{attr}.joblib')

    clf = joblib.load(clf_path)
    le  = joblib.load(le_path)

    y_pred_encoded = clf.predict(X)
    y_pred_labels  = le.inverse_transform(y_pred_encoded)

    periphery[attr] = y_pred_labels
    print(f"{attr}: {len(periphery):,} predictions  |  unique classes: {sorted(periphery[attr].unique())}")

policy_era: 279,793 predictions  |  unique classes: [np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
dilap_index: 279,793 predictions  |  unique classes: [np.float64(0.0), np.float64(1.0), np.float64(3.0), np.float64(6.0)]
landuse: 279,793 predictions  |  unique classes: [np.float64(1.0), np.float64(7.0)]


## 4 · Predict continuous attributes

In [11]:
reg_path = os.path.join(TRAINED_MODELS_DIR, 'regressor.joblib')
reg = joblib.load(reg_path)
print(f"Loaded: {reg_path}")

y_pred_cont = reg.predict(X)

for i, attr in enumerate(CONTINUOUS_TARGETS):
    periphery[attr] = y_pred_cont[:, i]
    print(f"{attr}: min={periphery[attr].min():.2f}  max={periphery[attr].max():.2f}  mean={periphery[attr].mean():.2f}")

Loaded: D:\SID\trained_models\regressor.joblib
mean_area: min=-242.92  max=4109.03  mean=1829.66
perimeter: min=19.93  max=167.33  mean=108.29
mean_pop: min=368.04  max=750.37  mean=430.86


## 5 · Save predictions

In [12]:
output_cols = [SEG_ID_COL] + CATEGORICAL_TARGETS + CONTINUOUS_TARGETS + ['whole_CH_2']
out = periphery[output_cols].copy()

out.to_csv(PREDICTIONS_JOINED_CSV, index=False)
print(f"Saved: {PREDICTIONS_JOINED_CSV}")
print(f"Shape: {out.shape}")
print(out.head())

Saved: D:\SID\data\outputs\predictions_joined.csv
Shape: (279793, 8)
   Depthmap_R  policy_era  dilap_index  landuse    mean_area   perimeter  \
0           1           4          0.0      1.0  1786.190038  104.668923   
1           4           4          0.0      1.0  1625.092281  103.274768   
2           8           4          0.0      1.0  1443.068968  101.783324   
3          12           4          0.0      1.0  1346.726247   99.486082   
4          13           5          0.0      1.0  1405.562793   98.898257   

     mean_pop  whole_CH_2  
0  415.702165           1  
1  411.520990           1  
2  410.779403          10  
3  400.433601          10  
4  399.555899          10  
